# HuggingFace Transformers 기초

> Phase 1에서 밑바닥부터 구현한 Transformer를 이제 **실전 라이브러리**로 다루기

In [2]:
# === 환경 설치 ===
# 최초 1회만 실행하면 됨 (이미 설치되어 있으면 스킵)
!pip install transformers torch

  Using cached pyyaml-6.0.3-cp310-cp310-win_amd64.whl.metadata (2.4 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer_slim-0.21.1-py3-none-any.whl.metadata (16 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached hf_xet-1.2.0-cp37-abi3-win_amd64.whl.metadata (5.0 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached anyio-4.12.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/10.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/10.3 MB ? eta -:--:--
   -- ------------------------------------- 0.5/10.3 MB 1.9 MB/s eta 0:00:06
   ---- -------------------------------

---
## 1. Auto 클래스: 모든 것의 시작점

- `AutoTokenizer` = Phase 1의 BasicTokenizer의 실전 버전
- `AutoModel` = Phase 1의 GPT 클래스의 실전 버전
- `AutoConfig` = Phase 1의 하이퍼파라미터 (n_head, n_layer 등)

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig

# === GPT-2 모델 설정 확인 ===
# Phase 1에서 직접 정한 n_head, n_layer 같은 값들이
# config.json에 저장되어 있다
config = AutoConfig.from_pretrained('gpt2')

print("GPT-2 모델 설정:")
print(f"  n_layer (Transformer 블록 수): {config.n_layer}")
print(f"  n_head (Attention 헤드 수): {config.n_head}")
print(f"  n_embd (임베딩 차원): {config.n_embd}")
print(f"  vocab_size (어휘 크기): {config.vocab_size}")
print(f"  n_positions (최대 시퀀스 길이): {config.n_positions}")

# Phase 1에서 우리가 정한 값과 비교:
# Phase 1: n_head=4, n_embd=64 (소규모)
# GPT-2:  n_head=12, n_embd=768 (실전 스케일)

c:\Users\jskim\anaconda3\envs\distillation\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPT-2 모델 설정:
  n_layer (Transformer 블록 수): 12
  n_head (Attention 헤드 수): 12
  n_embd (임베딩 차원): 768
  vocab_size (어휘 크기): 50257
  n_positions (최대 시퀀스 길이): 1024


c:\Users\jskim\anaconda3\envs\distillation\lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jskim\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [4]:
# === from_pretrained: 모델 이름만 주면 알아서 다운로드 ===
# HuggingFace Hub에서 config + 가중치 + 토크나이저를 가져온다
# 처음 실행 시 다운로드 되고, 이후에는 캐시에서 로드
tokenizer = AutoTokenizer.from_pretrained('gpt2')

print(f"토크나이저 타입: {type(tokenizer).__name__}")
print(f"어휘 크기: {tokenizer.vocab_size}")
print(f"\nPhase 1에서 우리가 만든 BasicTokenizer: vocab_size=276")
print(f"GPT-2 실전 토크나이저: vocab_size={tokenizer.vocab_size}")
print(f"-> 같은 BPE 원리, 스케일만 다름")

토크나이저 타입: GPT2Tokenizer
어휘 크기: 50257

Phase 1에서 우리가 만든 BasicTokenizer: vocab_size=276
GPT-2 실전 토크나이저: vocab_size=50257
-> 같은 BPE 원리, 스케일만 다름


---
## 2. 3가지 모델 구조 확인하기

In [5]:
from transformers import AutoModel, AutoModelForCausalLM, AutoModelForSequenceClassification

# === 3가지 모델 구조와 HuggingFace 클래스 매핑 ===
model_types = {
    "Encoder-only (BERT)": {
        "class": "AutoModel / AutoModelForSequenceClassification",
        "model": "bert-base-uncased",
        "desc": "양방향 Attention, 분류/유사도 특화",
    },
    "Decoder-only (GPT-2)": {
        "class": "AutoModelForCausalLM",
        "model": "gpt2",
        "desc": "Causal Mask 적용, 텍스트 생성 특화 (← Phase 1에서 만든 것)",
    },
    "Encoder-Decoder (T5)": {
        "class": "AutoModelForSeq2SeqLM",
        "model": "t5-small",
        "desc": "Encoder로 입력 이해, Decoder로 출력 생성",
    },
}

print("모델 구조 요약:")
print("=" * 70)
for name, info in model_types.items():
    print(f"\n{name}")
    print(f"  HF 클래스: {info['class']}")
    print(f"  대표 모델: {info['model']}")
    print(f"  특징: {info['desc']}")

모델 구조 요약:

Encoder-only (BERT)
  HF 클래스: AutoModel / AutoModelForSequenceClassification
  대표 모델: bert-base-uncased
  특징: 양방향 Attention, 분류/유사도 특화

Decoder-only (GPT-2)
  HF 클래스: AutoModelForCausalLM
  대표 모델: gpt2
  특징: Causal Mask 적용, 텍스트 생성 특화 (← Phase 1에서 만든 것)

Encoder-Decoder (T5)
  HF 클래스: AutoModelForSeq2SeqLM
  대표 모델: t5-small
  특징: Encoder로 입력 이해, Decoder로 출력 생성


In [6]:
# === 실제 모델 구조 살펴보기: GPT-2 ===
# Phase 1에서 만든 Block(Attention + FFN)이 실제로 어떻게 생겼는지 확인
model = AutoModelForCausalLM.from_pretrained('gpt2')

# 모델 구조 출력 (간략화)
print("GPT-2 모델 구조:")
print("=" * 50)
for name, param in model.named_parameters():
    # 첫 번째 Transformer 블록만 표시
    if 'h.0.' in name or 'wte' in name or 'wpe' in name or 'ln_f' in name:
        print(f"  {name}: {list(param.shape)}")

print(f"\n전체 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")
print(f"Phase 1 GPT: ~수백 파라미터")
print(f"GPT-2: ~124M 파라미터")
print(f"GPT-4: ~1.8T 파라미터 (추정)")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1113.36it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT-2 모델 구조:
  transformer.wte.weight: [50257, 768]
  transformer.wpe.weight: [1024, 768]
  transformer.h.0.ln_1.weight: [768]
  transformer.h.0.ln_1.bias: [768]
  transformer.h.0.attn.c_attn.weight: [768, 2304]
  transformer.h.0.attn.c_attn.bias: [2304]
  transformer.h.0.attn.c_proj.weight: [768, 768]
  transformer.h.0.attn.c_proj.bias: [768]
  transformer.h.0.ln_2.weight: [768]
  transformer.h.0.ln_2.bias: [768]
  transformer.h.0.mlp.c_fc.weight: [768, 3072]
  transformer.h.0.mlp.c_fc.bias: [3072]
  transformer.h.0.mlp.c_proj.weight: [3072, 768]
  transformer.h.0.mlp.c_proj.bias: [768]
  transformer.ln_f.weight: [768]
  transformer.ln_f.bias: [768]

전체 파라미터 수: 124,439,808
Phase 1 GPT: ~수백 파라미터
GPT-2: ~124M 파라미터
GPT-4: ~1.8T 파라미터 (추정)


In [8]:
# === Phase 1과의 레이어 대응 관계 ===
# Phase 1에서 만든 각 레이어가 GPT-2에서 어떤 이름인지 확인
mapping = [
    ("Phase 1", "GPT-2", "설명"),
    ("token_embedding", "wte (word token embedding)", "토큰 임베딩"),
    ("position_embedding", "wpe (word position embedding)", "위치 임베딩"),
    ("Block (sa + ffwd)", "h.0, h.1, ... h.11", "Transformer 블록 12개"),
    ("sa (self attention)", "h.X.attn", "Self-Attention"),
    ("c_attn (Q,K,V)", "h.X.attn.c_attn", "Q/K/V 생성 (3*n_embd)"),
    ("c_proj", "h.X.attn.c_proj", "Attention 출력 프로젝션"),
    ("ffwd", "h.X.mlp", "FeedForward 네트워크"),
    ("ln (LayerNorm)", "h.X.ln_1, h.X.ln_2", "레이어 정규화"),
    ("lm_head", "lm_head", "최종 어휘 크기로 프로젝션"),
]

header = "설명명"
print(f"{'Phase 1':<25} {'GPT-2':<35} {header}")
print("=" * 80)
for p1, gpt2, desc in mapping[1:]:
    print(f"  {p1:<23} {gpt2:<33} {desc}")

Phase 1                   GPT-2                               설명명
  token_embedding         wte (word token embedding)        토큰 임베딩
  position_embedding      wpe (word position embedding)     위치 임베딩
  Block (sa + ffwd)       h.0, h.1, ... h.11                Transformer 블록 12개
  sa (self attention)     h.X.attn                          Self-Attention
  c_attn (Q,K,V)          h.X.attn.c_attn                   Q/K/V 생성 (3*n_embd)
  c_proj                  h.X.attn.c_proj                   Attention 출력 프로젝션
  ffwd                    h.X.mlp                           FeedForward 네트워크
  ln (LayerNorm)          h.X.ln_1, h.X.ln_2                레이어 정규화
  lm_head                 lm_head                           최종 어휘 크기로 프로젝션


---
## 3. Model Card 읽기 실습

In [ ]:
# === VRAM 추정 공식 ===
# 필요 VRAM ≈ 파라미터 수 × 정밀도 바이트
models = [
    ("GPT-2", 124_000_000),
    ("Llama 3.1 8B", 8_000_000_000),
    ("Llama 3.1 70B", 70_000_000_000),
]

precisions = [
    ("FP32", 4),
    ("FP16/BF16", 2),
    ("INT8", 1),
    ("INT4", 0.5),
]

header = "모델"
print(f"{header:<20}", end="")
for prec_name, _ in precisions:
    print(f"{prec_name:>12}", end="")
print()
print("=" * 70)

for model_name, params in models:
    print(f"{model_name:<20}", end="")
    for prec_name, bytes_per_param in precisions:
        vram_gb = (params * bytes_per_param) / (1024**3)
        print(f"{vram_gb:>10.1f}GB", end="")
    print()

print("\n판단 기준:")
print("  RTX 3090/4090: 24GB VRAM")
print("  -> Llama 8B는 INT4로 로드 가능 (3.7GB)")
print("  -> Llama 70B는 INT4로도 32.6GB -> 단일 GPU 불가")

In [11]:
# === 실제 모델의 메모리 사용량 확인 ===
import torch

# GPT-2는 작아서 CPU에서도 로드 가능
model = AutoModelForCausalLM.from_pretrained('gpt2')

# 파라미터 메모리 계산
total_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
total_mb = total_bytes / (1024**2)

print(f"GPT-2 메모리 사용량:")
print(f"  파라미터 수: {sum(p.numel() for p in model.parameters()):,}")
print(f"  정밀도: {next(model.parameters()).dtype}")  # float32
print(f"  메모리: {total_mb:.1f}MB ({total_mb/1024:.2f}GB)")

# 메모리 정리
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 998.11it/s, Materializing param=transformer.wte.weight]              
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT-2 메모리 사용량:
  파라미터 수: 124,439,808
  정밀도: torch.float32
  메모리: 474.7MB (0.46GB)


---
## 4. 간단한 텍스트 생성 체험

In [12]:
# === GPT-2로 텍스트 생성 ===
# Phase 1에서 while문으로 다음 토큰 예측을 반복했던 것을
# generate() 한 줄로 대체
tokenizer = AutoTokenizer.from_pretrained('gpt2')
model = AutoModelForCausalLM.from_pretrained('gpt2')

# 토큰화 (Phase 1의 encode() 역할)
prompt = "The meaning of life is"
inputs = tokenizer(prompt, return_tensors='pt')  # PyTorch 텐서로 반환

print(f"입력: '{prompt}'")
print(f"input_ids: {inputs['input_ids']}")
print(f"attention_mask: {inputs['attention_mask']}")

# 생성 (Phase 1의 while문 대신)
with torch.no_grad():  # 추론 시 gradient 계산 불필요
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,   # 최대 50토큰 생성
        do_sample=True,      # 샘플링 활성화
        temperature=0.7,     # 적당한 창의성
        top_p=0.9,           # 상위 90% 확률의 토큰만 후보
    )

# 디코딩 (Phase 1의 decode() 역할)
generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\n생성 결과:\n{generated}")

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1093.29it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


입력: 'The meaning of life is'
input_ids: tensor([[ 464, 3616,  286, 1204,  318]])
attention_mask: tensor([[1, 1, 1, 1, 1]])

생성 결과:
The meaning of life is not what you think it is.

The meaning of life is not what you think it is. The meaning of life is not what you think it is. The meaning of life is not what you think it is.

The meaning of


In [13]:
# === Phase 1 vs Phase 2 전체 비교 요약 ===
print("프로세스 비교:")
print("=" * 60)
comparison = [
    ("토크나이저 정의", "BasicTokenizer 클래스 직접 구현", "AutoTokenizer.from_pretrained()"),
    ("인코딩", "tokenizer.encode(text)", "tokenizer(text, return_tensors='pt')"),
    ("모델 정의", "class GPT(nn.Module) 직접 구현", "AutoModelForCausalLM.from_pretrained()"),
    ("학습", "for문 + backward() + step()", "Trainer / SFTTrainer (Phase 4)"),
    ("생성", "while문으로 다음 토큰 반복", "model.generate()"),
    ("디코딩", "tokenizer.decode(ids)", "tokenizer.decode(ids)"),
]

for step, p1, p2 in comparison:
    print(f"\n  [{step}]")
    print(f"    Phase 1: {p1}")
    print(f"    Phase 2: {p2}")

# 메모리 정리
del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

프로세스 비교:

  [토크나이저 정의]
    Phase 1: BasicTokenizer 클래스 직접 구현
    Phase 2: AutoTokenizer.from_pretrained()

  [인코딩]
    Phase 1: tokenizer.encode(text)
    Phase 2: tokenizer(text, return_tensors='pt')

  [모델 정의]
    Phase 1: class GPT(nn.Module) 직접 구현
    Phase 2: AutoModelForCausalLM.from_pretrained()

  [학습]
    Phase 1: for문 + backward() + step()
    Phase 2: Trainer / SFTTrainer (Phase 4)

  [생성]
    Phase 1: while문으로 다음 토큰 반복
    Phase 2: model.generate()

  [디코딩]
    Phase 1: tokenizer.decode(ids)
    Phase 2: tokenizer.decode(ids)


---
## 정리

| 개념 | 핵심 |
|------|------|
| **Auto 클래스** | 모델 이름만 주면 알아서 맞는 구조 로드 |
| **from_pretrained** | Hub에서 config + 가중치 + 토크나이저 다운로드 |
| **3가지 구조** | Encoder(BERT), Decoder(GPT), Enc-Dec(T5) |
| **VRAM 추정** | 파라미터 수 × 정밀도 바이트 |